In [23]:
import torch
import numpy as np

from sklearn import metrics

from config import config_test

from encoder import GCNEncoder
from aggregator import MaxminMeanAggregator
from batch import load_val

import pickle

from random_walk import random_walking
from gene_position import get_gene_position

import os
import random
from efficient_apriori import apriori

from plot_results import plot_apriori
import json
from datetime import datetime

In [2]:
def model_pred(data, test_batchloader, model, Aggregator):
    model.eval()
    Aggregator.eval()
    with torch.no_grad():
        total_pred = []
        test_label = []
        num_data = 0

        v = model(data.x, data.edge_index, data.edge_attr)
        
        while True :
            hedges, labels, is_last = test_batchloader.next()
            batch_size = len(hedges)
            num_data+=batch_size
            for hedge in hedges :
                embeddings = v[hedge]
                #pred, _ = Aggregator(embeddings)
                max_min_dist = torch.max(hedge) - torch.min(hedge)
                pred, _ = Aggregator(max_min_dist,embeddings)
                total_pred.append(pred.detach())
                test_label.append(hedge.detach().cpu().tolist())
            
            if is_last :
                break
                
        total_pred = torch.stack(total_pred)       
        total_pred = total_pred.squeeze()

    return total_pred.tolist(), test_label

In [3]:
def generate_filter(points, graph_data, device):

    rands = random_walking(graph_data, points, device) ###
    test_batchloader = load_val(rands, 1000, device, label="pos")
    pred_score, samples = model_pred(graph_data, test_batchloader, model, Aggregator)

    label = []
    p = []
    for i, j in enumerate(pred_score):
        if  j > 0.88 and len(samples[i]) < 6 and len(samples[i]) > 2:
            p.append(j)
            label.append(samples[i])

    l = []
    for i in label:
        l.append(frozenset(i))
    l = list(set(l))

    label = []
    for i in l:
        label.append(list(i))

    return label

In [4]:
def process_apriori(point, gene_name, graph_data, device, results_dir="results",
                    min_support=0.001, min_confidence=0.8, epsilon=1,
                    extra_background_ratio=0.5, resample_support=0.0003,
                    resample_confidence=0.8, save_files=True):
    """
    封装的 Apriori 分析函数，支持结果缓存。

    Parameters:
        point (int): 输入的目标点。
        gene_name (str): 基因名称，用于生成文件名。
        graph_data (function): 子图数据。
        results_dir (str): 结果保存的目录。
        min_support (float): 初始支持度。
        min_confidence (float): 初始置信度。
        epsilon (float): 用于调整距离的常量。
        extra_background_ratio (float): 信号背景比例。
        resample_support (float): 重新筛选的支持度。
        resample_confidence (float): 重新筛选的置信度。
        save_files (bool): 是否保存生成的文件。

    Returns:
        dict: 包含结果的字典，包括信号、背景、混合数据和相关规则信息。
    """
    # 创建结果目录
    os.makedirs(results_dir, exist_ok=True)
    
    # 文件路径
    signal_file = os.path.join(results_dir, f"signal_{gene_name}_{point}.pkl")
    background_file = os.path.join(results_dir, f"background_{gene_name}_{point}.pkl")
    mix_file = os.path.join(results_dir, f"mix_{gene_name}_{point}.pkl")

    # 检查缓存是否存在
    if os.path.exists(signal_file) and os.path.exists(background_file) and os.path.exists(mix_file):
        print(f"Files for point {point} already exist. Loading from cache...")
        with open(signal_file, "rb") as f:
            signal = pickle.load(f)
        with open(background_file, "rb") as f:
            background = pickle.load(f)
            
        transactions = signal
        itemsets, rules = apriori(transactions, min_support=min_support, min_confidence=min_confidence)

        # 提取规则中的项目
        es = []
        for r in rules:
            if len(r.lhs) > 1:
                es.extend(list(r.lhs + r.rhs))

        es = list(set(es))

        if len(es) == 0:
            print(f"{point} no Apriori result.")
            return None

        # 确定区域范围
        area_min = max(0, min(es) - 10)
        area_max = min(max(es) + 10, 999)

    else:
        #try:
            # 生成信号
            signal = generate_filter([point], graph_data, device)

            # Apriori 初步分析
            transactions = signal
            itemsets, rules = apriori(transactions, min_support=min_support, min_confidence=min_confidence)

            # 提取规则中的项目
            es = []
            for r in rules:
                if len(r.lhs) > 1:
                    es.extend(list(r.lhs + r.rhs))

            es = list(set(es))

            if len(es) == 0:
                print(f"{point} no Apriori result.")
                return None

            # 确定区域范围
            area_min = max(0, min(es) - 10)
            area_max = min(max(es) + 10, 999)
            points = list(range(area_min, area_max + 1))

            # 生成背景
            background = generate_filter(points, graph_data, device)

            # 保存信号和背景
            if save_files:
                with open(signal_file, "wb") as f:
                    pickle.dump(signal, f)
                with open(background_file, "wb") as f:
                    pickle.dump(background, f)


#         except Exception as e:
#             print(f"Error processing point {point}: {e}")
#             return None

        
    # 计算背景抽样概率
    distances = [min([abs(point - x) for x in sublist]) for sublist in background]
    adjusted_distances = [d + epsilon for d in distances]
    probabilities = 1 / np.array(adjusted_distances)
    probabilities /= probabilities.sum()

    n = int(len(signal))
    sampled_indices = np.random.choice(len(background), size=n, replace=False, p=probabilities)
    sampled_lists = [background[i] for i in sampled_indices]

    # 生成混合交易数据
    transactions = signal + sampled_lists + random.sample(background, int(extra_background_ratio * len(signal)))

    if save_files:
        with open(mix_file, "wb") as f:
            pickle.dump(transactions, f)        

    # 再次运行 Apriori
    itemsets, rules = apriori(transactions, min_support=resample_support, min_confidence=resample_confidence)

    # 计算新的支持度
    count = [r.count_full for r in rules if len(r.lhs) > 1]
    count.sort(reverse=True)

    if len(count) > 12:
        resample_support = round(count[12] / len(transactions), 4)
        itemsets, rules = apriori(transactions, min_support=resample_support, min_confidence=resample_confidence)

    # 提取规则信息
    rs, es, confs, counts, convs = [], [], {}, {}, {}

    for r in rules:
        if len(r.lhs) > 1:
            if all(area_min <= x <= area_max for x in r.lhs + r.rhs):
                rs.append(list(r.lhs + r.rhs))
                es.extend(list(r.lhs + r.rhs))
                confs[tuple(r.lhs + r.rhs)] = r.confidence
                counts[tuple(r.lhs + r.rhs)] = r.count_full
                convs[tuple(r.lhs + r.rhs)] = r.conviction

    es = sorted(set(es))
    start_bin = max(0, min(es) - 5)
    end_bin = min(max(es) + 5, 999)

    return {
        "num_samples": len(transactions),
        "rules": rs,
        "bins": es,
        "counts": counts,
        "start_bin": start_bin,
        "end_bin": end_bin,
        "resample_support":resample_support, 
        "resample_confidence":resample_confidence
    }


In [22]:
def convert_keys_to_str(data):
    if isinstance(data, dict):
        return {str(key): convert_keys_to_str(value) for key, value in data.items()}
    elif isinstance(data, list):
        return [convert_keys_to_str(item) for item in data]
    else:
        return data

In [25]:
cell_line = 'K562'

model_dir = 'models/comprehensive/'

gene_name = 'MYB'

file_paths = {
    'atac': "/public/home/wuyy/K562_hg38/bigwigs/ENCFF754EAC.bigWig",
    'h3k27ac': "/public/home/wuyy/K562_hg38/bigwigs/ENCFF381NDD.bigWig",
    'h3k4me1': "/public/home/wuyy/K562_hg38/bigwigs/ENCFF761XBZ.bigWig",
    'ctcf': "/public/home/wuyy/K562_hg38/bigwigs/ENCFF675GVW.bigWig",
    'rad21': "/public/home/wuyy/K562_hg38/bigwigs/ENCFF652NKM.bigWig",
    'chrom_state': "/public/home/wuyy/K562_hg38/chrom_state/ENCFF319VXX.bigBed"
}

file_paths = False


def run_phoci(cell_line, model_dir, file_paths):

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    dim_vertex = 400
    input_dim = 15 #features_num
    bs = 512
    n_layers = 3

    model = GCNEncoder(input_dim, dim_vertex, dim_vertex, n_layers)
    model.to(device)
    model.load_state_dict(torch.load(model_dir+"model"))

    cls_layers = [dim_vertex*2+2, dim_vertex, 256, 128, 32, 8, 1] #[dim_vertex, 128, 8, 1]
    Aggregator = MaxminMeanAggregator(dim_vertex, cls_layers)#MaxminAggregator(dim_vertex, cls_layers)
    Aggregator.to(device)
    Aggregator.load_state_dict(torch.load(model_dir+"Aggregator"))

    subgraph_index, chrom, test_points, subgraph_table, gene_index = get_gene_position(gene_name, cell_line)

    with open("data/"+cell_line+"_hg38/input_graph/"+chrom+"_sliding_data","rb") as f:
        sliding_data = pickle.load(f)

    graph_data = sliding_data[subgraph_index]


    for point in test_points:

        res = process_apriori(point, gene_name, graph_data, device)


        start = subgraph_table[res['start_bin']:res['start_bin']+1].Start.values[0]
        end = subgraph_table[res['end_bin']:res['end_bin']+1].End.values[0]


        if file_paths:
            plot_apriori(point, graph_data, res['rules'], res['bins'], start, end, res['start_bin'], res['end_bin'], res['counts'],
                 file_paths, chrom, gene_index, gene_name, res['resample_support'], res['resample_confidence'], res['num_samples'])

        else:
            converted_data = convert_keys_to_str(res)

            # 保存为 JSON 文件
            # 获取当前时间
            now = datetime.now()

            # 格式化为字符串
            time_str = now.strftime("%Y-%m-%d_%H-%M-%S")
            file_name = "apriori_rules/apriori_"+gene_name+"_"+str(point)+"_"+time_str+".json"
            with open(file_name, 'w', encoding='utf-8') as f:
                json.dump(converted_data, f, ensure_ascii=False, indent=4)

            print(f"数据已成功保存到 {file_name}")
    

In [17]:
run_phoci(cell_line, model_dir, file_paths)


{'num_samples': 10850,
 'rules': [[34, 48, 2],
  [14, 18, 24],
  [15, 54, 24],
  [35, 76, 24],
  [40, 74, 24],
  [60, 79, 24]],
 'bins': [2, 14, 15, 18, 24, 34, 35, 40, 48, 54, 60, 74, 76, 79],
 'counts': {(34, 48, 2): 5,
  (14, 18, 24): 5,
  (15, 54, 24): 5,
  (35, 76, 24): 5,
  (40, 74, 24): 5,
  (60, 79, 24): 5},
 'start_bin': 0,
 'end_bin': 84,
 'resample_support': 0.0004,
 'resample_confidence': 0.8}